# FACULTY SOLUTION — Week KPI scoring & consequences: Seven-KPI financial state, composite scoring, and consequence catalog
Computes every answer from the canonical CSVs and asserts it against the golden fixtures.

In [ ]:
import pandas as pd, numpy as np, math, json, itertools
d = {s: pd.read_csv(f'../data/{s}.csv') for s in ['state_opening', 'kpi_definitions', 'kpi_rules', 'input_dictionary', 'reference_team_decisions', 'reference_team_inputs', 'consequence_catalog', 'consequence_resolution', 'worked_example_normalization']}

In [ ]:

def _state_run(d):
    import math
    P = {x.parameter: x.value for x in d['state_opening'].itertuples()}
    rules = d['kpi_rules']; teams = list(d['reference_team_inputs'].team.unique())
    inp = d['reference_team_inputs']
    state_vars = ['ebitda', 'capital_employed', 'net_debt', 'sustaining_capex', 'integrated_margin_per_boe',
                  'refining_vs_benchmark', 'nonfuel_per_site_k', 'asset_health']
    traj = {}
    for t in teams:
        s = {v: P[v] for v in state_vars}
        for wk in range(1, 14):
            rows = inp[(inp.team == t) & (inp.week == wk)]
            vals = dict(zip(rows.input_key, rows.value))
            for rr in rules[rules.week == wk].itertuples():
                if rr.input_key not in vals: raise KeyError(f'{t} week {wk} missing input {rr.input_key}')
                x = vals[rr.input_key]
                if rr.operation == 'add': s[rr.state_var] += rr.coefficient * x
                elif rr.operation == 'set': s[rr.state_var] = rr.coefficient * x
                else: raise ValueError(rr.operation)
            s['asset_health'] = min(100.0, max(0.0, s['asset_health']))
            traj[(t, wk)] = dict(s)
    return P, teams, traj

def _kpis(s, P):
    da = P['da_rate'] * s['capital_employed']; ebit = s['ebitda'] - da
    nopat = ebit * (1 - P['tax_rate'])
    return {'integrated_margin_per_boe': s['integrated_margin_per_boe'], 'roace': nopat / s['capital_employed'],
            'free_cash_flow': s['ebitda'] - P['tax_rate'] * ebit - s['sustaining_capex'],
            'refining_net_margin_vs_benchmark': s['refining_vs_benchmark'], 'retail_non_fuel_margin_per_site': s['nonfuel_per_site_k'],
            'net_debt_to_ebitda': s['net_debt'] / s['ebitda'], 'asset_health_index': s['asset_health']}

def _normalize(values, direction, scale=1.0):
    vals = {t: round(v * scale, 9) for t, v in values.items()}
    lo, hi = min(vals.values()), max(vals.values())
    if abs(hi - lo) < 1e-12: return {t: 50.0 for t in vals}
    return {t: 100.0 * ((v - lo) / (hi - lo) if direction == 'higher' else (hi - v) / (hi - lo)) for t, v in vals.items()}

def _rank(scores):
    order = sorted(scores, key=lambda t: -round(scores[t], 6)); ranks = {}
    for i, t in enumerate(order):
        ranks[t] = ranks[order[i - 1]] if i and round(scores[t], 6) == round(scores[order[i - 1]], 6) else i + 1
    return ranks

def _score_week(kp, kd, teams, scale_override=None):
    comp = {t: 0.0 for t in teams}
    for k in kd.itertuples():
        sc = (scale_override or {}).get(k.key, 1.0)
        norm = _normalize({t: kp[t][k.key] for t in teams}, k.direction, sc)
        for t in teams: comp[t] += k.weight * norm[t]
    return comp, _rank(comp)


def _resolve_consequences(d):
    dec = {}
    for row in d['reference_team_decisions'].itertuples():
        dec.setdefault(row.team, {})[row.decision_key] = row.value
    teams = list(dec); n = len(teams)
    util = sum(0.80 if dec[t]['run_rotterdam'] == '1' else 0.40 for t in teams) / n
    w1 = -9.0 * (util - 0.75)
    br = sum(dec[t]['fund_br'] == '1' for t in teams) / n
    w2 = -6.0 * (br - 0.5) if br > 0.5 else 3.0 * (0.5 - br)
    w3 = -0.10 * (sum(dec[t]['retail_match'] == '1' for t in teams) / n - 0.5)
    marg = sum(dec[t]['transfer_price_basis'] == 'marginal' for t in teams) / n
    mkt = sum(dec[t]['transfer_price_basis'] == 'market' for t in teams) / n
    cohort = 'disciplined' if marg >= 0.70 else ('lax' if mkt >= 0.70 else 'base')
    env = {'disciplined': 1520.0, 'base': 1150.0, 'lax': 950.0}[cohort]
    refv = {'market': -12.85, 'marginal': 42.15, 'lazy': 14.65}
    cov = {'cooperative': 0.70, 'obliged': 0.70, 'watchful': 0.45, 'guarded': 0.45, 'strained': 0.25, 'hostile': 0.25}
    out = {}
    for t in teams:
        x = dec[t]; helix = x['fund_helix'] == '1'
        whit = 'guarded' if x['transfer_price_basis'] == 'lazy' else 'cooperative'
        cover = 1 if refv[x['transfer_price_basis']] > 0 else 0
        canc = round((env - (880 if helix else 0)) * 0.30, 6)
        outlay = 640 * (x['fund_br'] == '1') + 400 * (x['fund_rot'] == '1') + 880 * helix
        cash = round(250 + 0.25 * float(x['ebitda_effect_musd']) - 0.10 * (outlay + 600 * (x['capacity_match'] == '1') + float(x['rebrand_cost_musd'])), 6)
        binding = int(canc < 200) + int(cov[whit] < 0.50) + cover + int(x['straits_pacific_standing'] in ('strained', 'hostile')) + int(cash < 150)
        vals = {'week3_window1_nwe_crack_shift': round(w1, 6), 'week4_cohort_state': cohort, 'week4_capital_envelope_musd': env,
                'week4_tp_delacroix_cover': cover, 'week4_whitaker_standing': whit, 'week5_hedge_coverage': cov[whit],
                'week6_cancellable_capex_musd': canc, 'week6_window2_crack_shift': round(w2, 6), 'week7_window3_nonfuel_shift': round(w3, 6),
                'week8_cash_cushion_musd': cash, 'week10_binding_count': binding,
                'week1_delacroix_initial_standing': 'cooperative' if x['first_meeting'] == 'delacroix' else 'guarded'}
        for k, v in vals.items(): out[(t, k)] = v
    return out

def compute(d):
    P, teams, traj = _state_run(d)
    kd = d['kpi_definitions']
    r = {'weights_sum': float(kd.weight.sum())}
    o = _kpis({v: P[v] for v in ['ebitda', 'capital_employed', 'net_debt', 'sustaining_capex', 'integrated_margin_per_boe',
                                  'refining_vs_benchmark', 'nonfuel_per_site_k', 'asset_health']}, P)
    for k, v in o.items(): r['opening_' + k] = v
    all_available = True; ranks_by_week = {}
    for wk in range(1, 14):
        kp = {t: _kpis(traj[(t, wk)], P) for t in teams}
        comp, rk = _score_week(kp, kd, teams)
        comp_s, rk_s = _score_week(kp, kd, teams, {'free_cash_flow': 1000.0, 'net_debt_to_ebitda': 0.001})
        r[f'w{wk}_rank_unit_invariant'] = all(rk[t] == rk_s[t] for t in teams)
        ranks_by_week[wk] = rk
        for t in teams:
            for k, v in kp[t].items():
                if v is None or (isinstance(v, float) and v != v): all_available = False
                r[f'{t}_w{wk}_{k}'] = v
            r[f'{t}_w{wk}_composite'] = comp[t]; r[f'{t}_w{wk}_rank'] = rk[t]
    r['all_kpis_available_all_weeks'] = all_available
    r['week1_all_tied'] = len(set(ranks_by_week[1].values())) == 1
    r['final_leader'] = min(teams, key=lambda t: (ranks_by_week[13][t], t))
    # consequence resolution for reference teams
    derived = _resolve_consequences(d); mism = []
    for row in d['consequence_resolution'].itertuples():
        v = derived[(row.team, row.consequence_key)]
        r[f'csq_{row.team}_{row.consequence_key}'] = v
        exp = row.expected_target_value
        ok = (str(v) == str(exp)) if isinstance(v, str) else abs(float(v) - float(exp)) < 1e-6
        if not ok: mism.append(f'{row.team}:{row.consequence_key}')
    r['consequences_reproduced'] = not mism
    inp = d['reference_team_inputs']
    r['binding_inputs_consistent'] = all(abs(float(inp[(inp.team == t) & (inp.input_key == 'i10_binding_count')].value.iloc[0]) - derived[(t, 'week10_binding_count')]) < 1e-9 for t in set(inp.team))
    r['opening_da'] = P['da_rate'] * P['capital_employed']
    # window-2 shift resolved from reference section's Week 6 decisions (cross-check against kpi inputs)
    w6 = d['reference_team_inputs']; br = w6[(w6.week == 6) & (w6.input_key == 'i6_br_upgrade_funded')]
    share = float(br.value.mean())
    shift = -6.0 * (share - 0.5) if share > 0.5 else 3.0 * (0.5 - share)
    r['section_br_upgrade_share'] = share; r['section_window2_shift'] = shift
    w8 = w6[(w6.week == 8) & (w6.input_key == 'i8_window2_shift')]
    r['window2_inputs_consistent'] = bool((abs(w8.value - shift) < 1e-9).all())
    return r


In [ ]:
fx = json.load(open('../fixtures/kpi_consequence_golden.json'))
r = compute(d)
bad = []
for k, v in fx['results'].items():
    a = r[k]
    ok = (str(a) == str(v)) if isinstance(v, str) else math.isclose(float(a), float(v), rel_tol=1e-3, abs_tol=1e-5)
    print(('OK  ' if ok else 'DIFF'), f'{k:36s}', a)
    if not ok: bad.append(k)
assert not bad, bad
print('FACULTY NOTEBOOK REPRODUCES GOLDEN FIXTURES: PASS')